# pragma-ai · Quickstart

**Atomic-fact reasoning over a knowledge graph. No vector database.**

This notebook walks you through installing pragma, ingesting a real research paper (PDF), and running 5 queries that demonstrate:

1. **Single-fact lookup** — "what is X?"
2. **Multi-hop reasoning** — facts assembled from multiple documents
3. **Explainability** — every answer cites the exact `fact_id` it depends on
4. **Token efficiency** — ~6× fewer tokens than vector RAG
5. **Streaming** — token-by-token answers via `kb.stream(...)`

Runtime: **CPU is enough.** No GPU, no vector DB, no external services.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kbpr21/pragma-ai/blob/main/docs/quickstart_colab.ipynb) · [GitHub](https://github.com/kbpr21/pragma-ai) · [PyPI](https://pypi.org/project/pragma-ai/)

## 1 • Install pragma + a free Groq API key

Get a free Groq key in <30 seconds: <https://console.groq.com/keys>. Free tier is plenty for this notebook.

In [ ]:
%pip install -q "pragma-ai[pdf]"

In [ ]:
import getpass, os

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("GROQ_API_KEY: ")

## 2 • Build a knowledge base from a real PDF

We'll ingest the **„Attention Is All You Need“** paper (the original Transformer paper, Vaswani et al., 2017). Replace the URL below with any PDF you like.

In [ ]:
import urllib.request
from pathlib import Path

PDF_URL = "https://arxiv.org/pdf/1706.03762"
PDF_PATH = Path("attention.pdf")

if not PDF_PATH.exists():
    urllib.request.urlretrieve(PDF_URL, PDF_PATH)

print(f"Downloaded {PDF_PATH} ({PDF_PATH.stat().st_size / 1024:.0f} KB)")

In [ ]:
from pragma import KnowledgeBase
from pragma.llm import get_provider

llm = get_provider("groq")
kb = KnowledgeBase(llm=llm, kb_dir="./pragma_kb")

result = kb.ingest(str(PDF_PATH), show_progress=True)
print(result.summary())

In [ ]:
stats = kb.stats()
print(f"documents={stats.documents}  facts={stats.facts}  entities={stats.entities}  edges={stats.relationships}")

## 3 • Five demo queries

Each call returns a `PragmaResult` with:

* `answer` — the synthesized natural-language answer
* `reasoning_path` — cited fact-by-fact reasoning chain
* `source_facts` — the atomic facts that were assembled
* `confidence` — aggregate confidence over cited facts
* `tokens_used`, `latency_ms`, `subgraph_size` — telemetry

In [ ]:
def show(question: str, **kw):
    print(f"\n▶ {question}")
    r = kb.query(question, **kw)
    print(f"  answer       : {r.answer}")
    print(f"  confidence   : {r.confidence:.2f}")
    print(f"  tokens used  : {r.tokens_used}   (vector RAG would use ~3000)")
    print(f"  latency      : {r.latency_ms:.0f} ms")
    print(f"  subgraph     : {r.subgraph_size} nodes/edges")
    if r.reasoning_path:
        print("  reasoning    :")
        for s in r.reasoning_path:
            print(f"    [{s.fact_id[:8]}] {s.explanation}")
    return r

In [ ]:
# 1. Single-fact lookup
_ = show("What is the Transformer architecture?")

In [ ]:
# 2. Multi-hop: requires combining at least two facts
_ = show("What problem does multi-head attention solve, and how is it different from single-head attention?")

In [ ]:
# 3. Explainability: inspect the cited facts directly
result = show("What is the dimensionality used for the model?")
print("\nSource facts cited:")
for f in result.source_facts:
    print(f"  {f.subject_id} | {f.predicate} | {f.object_value or f.object_id} (conf={f.confidence:.2f})")

In [ ]:
# 4. Token efficiency: pragma queries cost ~280 tokens vs ~3000 for vector RAG
_ = show("Compare encoder and decoder stacks in the Transformer.")

In [ ]:
# 5. Streaming: tokens arrive one at a time (great for UX)
import asyncio, sys

async def stream_demo():
    async for tok in kb.stream("Why was scaled dot-product attention chosen?"):
        sys.stdout.write(tok); sys.stdout.flush()
    print()

await stream_demo()

## 4 • What to try next

* Replace `attention.pdf` with **multiple PDFs** in a folder — `kb.ingest("./papers/")`.
* Inspect entities the extractor found: `pragma entities --limit 50` (CLI) or `kb._storage.get_all_entities()`.
* Run the offline **evaluator** with your own ground-truth questions:

  ```python
  from pragma.eval import Evaluator, TestCase
  cases = [TestCase(query="...", expected_answer_contains=["..."])]
  print(Evaluator(kb, cases).run().summary())
  ```
* Override any built-in prompt without touching code: set `PRAGMA_PROMPT_FACT_EXTRACTION=/path/to/your.txt`.

**⭐ If pragma was useful, please [star the repo](https://github.com/kbpr21/pragma-ai).**

In [ ]:
kb.close()